<a href="https://colab.research.google.com/github/jasjot06/Clinical-Sentinel-LAOS/blob/main/notebooka0b062f549.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

jasjotsingh06_clinical_guidelines_rag_path = kagglehub.dataset_download('jasjotsingh06/clinical-guidelines-rag')
jasjotsingh06_chroma_dbbackup_path = kagglehub.dataset_download('jasjotsingh06/chroma-dbbackup')

print('Data source import complete.')


In [ ]:
# Cell 1 — Install all packages (fixed)
import subprocess

packages = [
    "transformers>=4.40.0",
    "peft",
    "accelerate",
    "bitsandbytes",
    "datasets",
    "trl",
    "openai-whisper",
    "pyannote.audio",
    "chromadb",
    "sentence-transformers",
    "pypdf",
    "evaluate",
    "bert-score",
    "soundfile",
    "librosa",
    "tqdm",
    "jiwer",          # WER calculation
    "gtts",           # Google TTS — works on Python 3.12, replaces Coqui TTS
    "pydub",          # audio processing
]

failed = []
for pkg in packages:
    result = subprocess.run(
        ["pip", "install", "-q", pkg],
        capture_output=True
    )
    if result.returncode != 0:
        failed.append(pkg)
        print(f"  FAILED: {pkg}")
    else:
        print(f"  OK: {pkg}")

if failed:
    print(f"\nFailed packages: {failed}")
else:
    print("\nAll packages installed successfully")

  OK: transformers>=4.40.0
  OK: peft
  OK: accelerate
  OK: bitsandbytes
  OK: datasets
  OK: trl
  OK: openai-whisper
  OK: pyannote.audio
  OK: chromadb
  OK: sentence-transformers
  OK: pypdf
  OK: evaluate
  OK: bert-score
  OK: soundfile
  OK: librosa
  OK: tqdm
  OK: jiwer
  OK: gtts
  OK: pydub

All packages installed successfully


In [ ]:
# Cell 2 — GPU check
import torch

print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory/1e9:.1f} GB")

print(f"\nTotal VRAM: {sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count()))/1e9:.1f} GB")

GPUs available: 2
  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB

Total VRAM: 31.3 GB


In [ ]:
# Cell 3 — Create directory structure
import os

dirs = [
    "/kaggle/working/data/raw_audio",
    "/kaggle/working/data/transcripts",
    "/kaggle/working/data/diarized",
    "/kaggle/working/data/synthesized_audio/doctor",
    "/kaggle/working/data/synthesized_audio/patient",
    "/kaggle/working/data/training",
    "/kaggle/working/data/evaluation",
    "/kaggle/working/rag/chroma_db",
    "/kaggle/working/models/whisper_finetuned",
    "/kaggle/working/models/lora_adapters",
    "/kaggle/working/outputs/reports",
]

for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Directory structure created:")
for d in dirs:
    print(f"  {d}")

Directory structure created:
  /kaggle/working/data/raw_audio
  /kaggle/working/data/transcripts
  /kaggle/working/data/diarized
  /kaggle/working/data/synthesized_audio/doctor
  /kaggle/working/data/synthesized_audio/patient
  /kaggle/working/data/training
  /kaggle/working/data/evaluation
  /kaggle/working/rag/chroma_db
  /kaggle/working/models/whisper_finetuned
  /kaggle/working/models/lora_adapters
  /kaggle/working/outputs/reports


In [ ]:
# Cell 3b — Restore Chroma DB from backup (fixed)
import shutil, os

SOURCE = "/kaggle/input/datasets/jasjotsingh06/chroma-dbbackup"
DEST = "/kaggle/working/rag/chroma_db"

os.makedirs(DEST, exist_ok=True)

# Copy entire Chroma DB structure to writable location
shutil.copytree(SOURCE, DEST, dirs_exist_ok=True)
print("Chroma DB copied to working directory")

# Verify
files = []
for root, dirs, fs in os.walk(DEST):
    for f in fs:
        files.append(os.path.join(root, f))
print(f"Files copied: {len(files)}")

Chroma DB copied to working directory
Files copied: 17


In [ ]:
# Cell 3c — Load embedder and verify collections
import chromadb
from sentence_transformers import SentenceTransformer

CHROMA_PATH = "/kaggle/working/rag/chroma_db"
DOMAINS = ["ophthalmology", "cardiology", "dermatology", "coding"]

print("Loading BGE embedder on CPU...")
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cpu")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collections = {}
for domain in DOMAINS:
    collections[domain] = chroma_client.get_or_create_collection(
        name=domain,
        metadata={"hnsw:space": "cosine"}
    )
    print(f"  {domain}: {collections[domain].count()} chunks")

Loading BGE embedder on CPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

  ophthalmology: 763 chunks
  cardiology: 989 chunks
  dermatology: 250 chunks
  coding: 223 chunks


In [ ]:
# Cell 4 — Define all 16 guideline files
import os

GUIDELINES_DIR = None
for dirname, _, filenames in os.walk('/kaggle/input'):
    if "Cataract in the Adult Eye PPP_7.9.25.pdf" in filenames:
        GUIDELINES_DIR = dirname
        break

if not GUIDELINES_DIR:
    raise FileNotFoundError("❌ Could not find the PDFs. Please check if the dataset is attached.")

print(f"✅ Found files at: {GUIDELINES_DIR}")

GUIDELINE_FILES = {
    "oph_cataract":       "Cataract in the Adult Eye PPP_7.9.25.pdf",
    "oph_glaucoma":       "Primary Open-Angle Glaucoma PPP_2026.pdf",
    "oph_diabetic_ret":   "Diabetic Retinopathy PPP_8.4.25.pdf",
    "oph_eye_eval":       "Comprehensive Adult Medical Eye Evaluation PPP_2026.pdf",
    "oph_refractive":     "Refractive Errors PPP_Updated 2025.pdf",
    "card_heart_failure": "heidenreich-et-al-2022-2022-aha-acc-hfsa-guideline-for-the-management-of-heart-failure-a-report-of-the-american-college.pdf",
    "card_hypertension":  "jones-et-al-2025-2025-aha-acc-aanp-aapa-abc-accp-acpm-ags-ama-aspc-nma-pcna-sgim-guideline-for-the-prevention-detection.pdf",
    "card_coronary":      "virani-et-al-2023-2023-aha-acc-accp-aspc-nla-pcna-guideline-for-the-management-of-patients-with-chronic-coronary.pdf",
    "card_acs":           "rao-et-al-2025-2025-acc-aha-acep-naemsp-scai-guideline-for-the-management-of-patients-with-acute-coronary-syndromes-a.pdf",
    "derm_melanoma":      "PIIS019096221832588X.pdf",
    "derm_bcc":           "PIIS019096221732529X.pdf",
    "derm_acne":          "PIIS0190962223033893.pdf",
    "derm_atopic":        "PIIS0190962226003439.pdf",
    "code_icd10":         "ICD-10-CM-Guidelines-April-1 FY2024.pdf",
    "code_em":            "mln006764_evaluation_management_services_0.pdf",
    "code_terminology":   "nrces_Clinical Terminology and Coding Standards.pdf",
}

DOMAIN_MAP = {
    "oph_cataract":       "ophthalmology",
    "oph_glaucoma":       "ophthalmology",
    "oph_diabetic_ret":   "ophthalmology",
    "oph_eye_eval":       "ophthalmology",
    "oph_refractive":     "ophthalmology",
    "card_heart_failure": "cardiology",
    "card_hypertension":  "cardiology",
    "card_coronary":      "cardiology",
    "card_acs":           "cardiology",
    "derm_melanoma":      "dermatology",
    "derm_bcc":           "dermatology",
    "derm_acne":          "dermatology",
    "derm_atopic":        "dermatology",
    "code_icd10":         "coding",
    "code_em":            "coding",
    "code_terminology":   "coding",
}

DOMAINS = ["ophthalmology", "cardiology", "dermatology", "coding"]

print("Verifying guideline files...")
missing = []
for key, filename in GUIDELINE_FILES.items():
    path = os.path.join(GUIDELINES_DIR, filename)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {key}")
    if status == "MISSING":
        missing.append(key)

print(f"\nFound: {len(GUIDELINE_FILES)-len(missing)}/16")

✅ Found files at: /kaggle/input/datasets/jasjotsingh06/clinical-guidelines-rag
Verifying guideline files...
  [OK] oph_cataract
  [OK] oph_glaucoma
  [OK] oph_diabetic_ret
  [OK] oph_eye_eval
  [OK] oph_refractive
  [OK] card_heart_failure
  [OK] card_hypertension
  [OK] card_coronary
  [OK] card_acs
  [OK] derm_melanoma
  [OK] derm_bcc
  [OK] derm_acne
  [OK] derm_atopic
  [OK] code_icd10
  [OK] code_em
  [OK] code_terminology

Found: 16/16


In [ ]:
# Cell 5 — Extract text from all 16 PDFs
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path, key):
    reader = PdfReader(pdf_path)
    pages = []
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if not text or len(text.strip()) < 50:
            continue
        pages.append({
            "key": key,
            "domain": DOMAIN_MAP.get(key, "general"),
            "source_file": os.path.basename(pdf_path),
            "page_number": page_num + 1,
            "text": text.strip()
        })
    return pages

def extract_all_guidelines():
    all_pages = []
    for key, filename in GUIDELINE_FILES.items():
        full_path = os.path.join(GUIDELINES_DIR, filename)
        if not os.path.exists(full_path):
            print(f"  SKIPPING {key}")
            continue
        try:
            pages = extract_text_from_pdf(full_path, key)
            all_pages.extend(pages)
            print(f"  {key}: {len(pages)} pages")
        except Exception as e:
            print(f"  ERROR {key}: {e}")
    print(f"\nTotal pages: {len(all_pages)}")
    return all_pages

all_pages = extract_all_guidelines()

  oph_cataract: 126 pages
  oph_glaucoma: 103 pages
  oph_diabetic_ret: 88 pages
  oph_eye_eval: 35 pages
  oph_refractive: 59 pages
  card_heart_failure: 138 pages
  card_hypertension: 105 pages
  card_coronary: 111 pages
  card_acs: 92 pages
  derm_melanoma: 43 pages
  derm_bcc: 20 pages
  derm_acne: 30 pages
  derm_atopic: 26 pages
  code_icd10: 120 pages
  code_em: 30 pages
  code_terminology: 66 pages

Total pages: 1192


In [ ]:
# Cell 6 — Chunk pages into 512-token chunks (LAOS standard)
def chunk_text(text, chunk_size=512, overlap=50):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

def chunk_all_pages(all_pages):
    all_chunks = []
    for page in all_pages:
        chunks = chunk_text(page["text"])
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "chunk_id": f"{page['key']}_p{page['page_number']}_c{i}",
                "key": page["key"],
                "domain": page["domain"],
                "source_file": page["source_file"],
                "page_number": page["page_number"],
                "chunk_index": i,
                "text": chunk,
            })
    print(f"Total chunks: {len(all_chunks)}")
    avg_len = sum(len(c["text"].split()) for c in all_chunks) // len(all_chunks)
    print(f"Average chunk length: {avg_len} words")
    return all_chunks

all_chunks = chunk_all_pages(all_pages)

Total chunks: 2225
Average chunk length: 376 words


In [ ]:
# Cell 7 — Build Chroma vector DB with BGE-Large-En embeddings

import chromadb
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

CHROMA_PATH = "/kaggle/working/rag/chroma_db"

print("Loading BGE-Large-En on CPU...")
embedder = SentenceTransformer("BAAI/bge-large-en-v1.5", device="cpu")
print("Embedder loaded")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collections = {}
for domain in DOMAINS:
    collections[domain] = chroma_client.get_or_create_collection(
        name=domain,
        metadata={"hnsw:space": "cosine"}
    )

BATCH_SIZE = 32
by_domain = {d: [] for d in DOMAINS}
for chunk in all_chunks:
    if chunk["domain"] in by_domain:
        by_domain[chunk["domain"]].append(chunk)

for domain, chunks in by_domain.items():
    if not chunks:
        continue
    print(f"\nEmbedding {len(chunks)} chunks — {domain}")
    collection = collections[domain]
    for i in tqdm(range(0, len(chunks), BATCH_SIZE)):
        batch = chunks[i: i+BATCH_SIZE]
        texts = [f"Represent this clinical guideline passage: {c['text']}" for c in batch]
        ids = [c["chunk_id"] for c in batch]
        metadatas = [{
            "key": c["key"],
            "source_file": c["source_file"],
            "page_number": c["page_number"],
            "domain": c["domain"],
        } for c in batch]
        embeddings = embedder.encode(
            texts, normalize_embeddings=True
        ).tolist()
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=[c["text"] for c in batch],
            metadatas=metadatas
        )
    print(f"  {domain}: {collection.count()} chunks stored")

print("\nChroma DB built successfully")

Loading BGE-Large-En on CPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

  cardiology: 989 chunks stored

Embedding 250 chunks — dermatology


100%|██████████| 8/8 [07:48<00:00, 58.57s/it]


  dermatology: 250 chunks stored

Embedding 223 chunks — coding


100%|██████████| 7/7 [06:02<00:00, 51.75s/it]

  coding: 223 chunks stored

Chroma DB built successfully


In [ ]:
# Cell 8 — RAG retrieve function + test
def retrieve(query, domain, top_k=5):
    if domain not in collections:
        domain = "coding"
    collection = collections[domain]
    query_embedding = embedder.encode(
        f"Represent this clinical query: {query}",
        normalize_embeddings=True
    ).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        retrieved.append({
            "text": doc,
            "source": meta["source_file"],
            "page": meta["page_number"],
            "score": round(1 - dist, 4)
        })
    return retrieved

# Test retrieval
test_queries = {
    "ophthalmology": "IOP threshold for glaucoma diagnosis",
    "cardiology":    "STEMI door to balloon time targets",
    "dermatology":   "ABCDE melanoma criteria",
    "coding":        "ICD-10 bilateral eye condition coding rules"
}

print("Testing RAG retrieval...\n")
for domain, query in test_queries.items():
    results = retrieve(query, domain, top_k=1)
    print(f"{domain.upper()}")
    print(f"  Query: {query}")
    print(f"  Score: {results[0]['score']} | {results[0]['source']} p.{results[0]['page']}")
    print(f"  Text: {results[0]['text'][:150]}\n")

Testing RAG retrieval...

OPHTHALMOLOGY
  Query: IOP threshold for glaucoma diagnosis
  Score: 0.7315 | Primary Open-Angle Glaucoma PPP_2026.pdf p.79
  Text: and ai development. Prog Retin Eye Res 2022;90:101052. 235. A nderson DR, Patella VM. Automated static perimetry, 2nd ed. St. Louis, MO: Mosby , 1999:

CARDIOLOGY
  Query: STEMI door to balloon time targets
  Score: 0.7394 | rao-et-al-2025-2025-acc-aha-acep-naemsp-scai-guideline-for-the-management-of-patients-with-acute-coronary-syndromes-a.pdf p.73
  Text: 2015. Circulation. 2017;136:1908–1919. 7 . Jacobs AK, Ali MJ, Best PJ, et al. Systems of care for ST-segment-elevation myocardial infarction: a policy

DERMATOLOGY
  Query: ABCDE melanoma criteria
  Score: 0.7088 | PIIS019096221832588X.pdf p.8
  Text: PATHOLOGY REPORT When a biopsy of a lesion clinically suggestive of primary CM is performed, the WG recommends that pertinent information be provided 

CODING
  Query: ICD-10 bilateral eye condition coding rules
  Score: 0.803 | I

In [ ]:
# Cell 9 — Save Chroma DB
import shutil

shutil.make_archive(
    "/kaggle/working/chroma_db_backup",
    "zip",
    "/kaggle/working/rag/chroma_db"
)
size = os.path.getsize("/kaggle/working/chroma_db_backup.zip")
print(f"Chroma DB saved: {size/1e6:.1f} MB")
print("Download from Output tab and upload as Kaggle dataset")

Chroma DB saved: 30.9 MB
Download from Output tab and upload as Kaggle dataset


In [ ]:
# Cell 10 — Download all text datasets
from datasets import load_dataset
import subprocess

print("Downloading MTS-Dialog...")
mts = load_dataset("har1/MTS_Dialogue-Clinical_Note")
print(f"MTS-Dialog: {len(mts['train'])} train examples")
print(f"Columns: {mts['train'].column_names}")

print("\nDownloading NoteChat...")
notechat = load_dataset("akemiH/NoteChat")
print(f"NoteChat: {len(notechat['train'])} train examples")
print(f"Columns: {notechat['train'].column_names}")

print("\nDownloading ACI-Bench...")
subprocess.run([
    "git", "clone",
    "https://github.com/wyim/aci-bench.git",
    "/kaggle/working/data/raw_datasets/aci-bench"
], check=True)
print("ACI-Bench downloaded")

README.md: 0.00B [00:00, ?B/s]

MTS-Dialog-TrainingSet%20%28SDHP%29.csv: 0.00B [00:00, ?B/s]

(…)Dialog-Validation%20Set%20%28SDHP%29.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1301 [00:00<?, ? examples/s]

MTS-Dialog: 1301 train examples
Columns: ['ID', 'section_header', 'section_text', 'dialogue']



...

gpt-3.5.csv:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/207001 [00:00<?, ? examples/s]

NoteChat: 207001 train examples
Columns: ['data', 'conversation']



Cloning into '/kaggle/working/data/raw_datasets/aci-bench'...


ACI-Bench downloaded


In [ ]:
# Cell 11 — Convert MTS-Dialog to authority schema
def convert_mts_dialog(example):
    dialogue_raw = example["dialogue"]
    note = example["section_text"]
    lines = dialogue_raw.strip().split("\n")
    doctor_utts, patient_utts, full_lines = [], [], []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.lower().startswith("doctor:"):
            text = line[7:].strip()
            doctor_utts.append(text)
            full_lines.append(f"[DOCTOR]: {text}")
        elif line.lower().startswith("patient:"):
            text = line[8:].strip()
            patient_utts.append(text)
            full_lines.append(f"[PATIENT]: {text}")
        else:
            if full_lines:
                full_lines[-1] += " " + line
    return {
        "source": "mts_dialog",
        "full_dialogue": "\n".join(full_lines),
        "doctor_utterances": " ".join(doctor_utts),
        "patient_utterances": " ".join(patient_utts),
        "target_note": note,
        "doctor_is_ground_truth": True
    }

mts_converted = []
for ex in mts["train"]:
    try:
        c = convert_mts_dialog(ex)
        if c["doctor_utterances"] and c["patient_utterances"]:
            mts_converted.append(c)
    except:
        continue

print(f"MTS-Dialog converted: {len(mts_converted)} examples")
print(f"\nSample dialogue:\n{mts_converted[0]['full_dialogue'][:400]}")
print(f"\nSample note:\n{mts_converted[0]['target_note'][:300]}")

MTS-Dialog converted: 1183 examples

Sample dialogue:
[DOCTOR]: What brings you back into the clinic today, miss?
[PATIENT]: I came in for a refill of my blood pressure medicine.
[DOCTOR]: It looks like Doctor Kumar followed up with you last time regarding your hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis and kidney stones.  Have you noticed any changes or do you have any concerns regarding these issues?
[PATIENT]: No

Sample note:
Symptoms: no fever, no chills, no cough, no congestion, no nausea, no vomiting, no chest pain, no chest pressure.
Diagnosis: hypertension, osteoarthritis, osteoporosis, hypothyroidism, allergic rhinitis, kidney stones
History of Patient: 76-year-old white female, presents to the clinic today origina


In [ ]:
# Cell 12 — Convert NoteChat to authority schema
def convert_notechat(example):
    conversation_raw = example["conversation"]
    note = example["data"]
    lines = conversation_raw.strip().split("\n")
    doctor_utts, patient_utts, full_lines = [], [], []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.lower().startswith("doctor:"):
            text = line[7:].strip()
            doctor_utts.append(text)
            full_lines.append(f"[DOCTOR]: {text}")
        elif line.lower().startswith("patient:"):
            text = line[8:].strip()
            patient_utts.append(text)
            full_lines.append(f"[PATIENT]: {text}")
    return {
        "source": "notechat",
        "full_dialogue": "\n".join(full_lines),
        "doctor_utterances": " ".join(doctor_utts),
        "patient_utterances": " ".join(patient_utts),
        "target_note": note,
        "doctor_is_ground_truth": True
    }

# Use 2000 from NoteChat
notechat_subset = notechat["train"].select(range(2000))
notechat_converted = []
for ex in notechat_subset:
    try:
        c = convert_notechat(ex)
        if c["doctor_utterances"] and c["patient_utterances"]:
            notechat_converted.append(c)
    except:
        continue

print(f"NoteChat converted: {len(notechat_converted)} examples")
print(f"\nSample dialogue:\n{notechat_converted[0]['full_dialogue'][:400]}")

NoteChat converted: 1961 examples

Sample dialogue:
[DOCTOR]: Hi, Mr. X, I'm Dr. Y. How are you feeling today?
[PATIENT]: Not too good, doctor. I've been feeling really sick lately.
[DOCTOR]: I understand. Can you tell me what symptoms you're experiencing?
[PATIENT]: Yes, I've been having a fever, a dry cough, and dyspnea.
[DOCTOR]: I see. You were hospitalized due to moderate ARDS from COVID-19, is that correct?
[PATIENT]: Yes, that's correct.
[DO


In [ ]:
# Cell 13 — Convert ACI-Bench to authority schema
import pandas as pd

aci_df = pd.read_csv(
    "/kaggle/working/data/raw_datasets/aci-bench/data/challenge_data/train.csv"
)
print(f"ACI-Bench columns: {aci_df.columns.tolist()}")
print(f"ACI-Bench size: {len(aci_df)}")
print(f"\nSample row:")
for col in aci_df.columns:
    print(f"  [{col}]: {str(aci_df.iloc[0][col])[:200]}")

ACI-Bench columns: ['dataset', 'encounter_id', 'dialogue', 'note']
ACI-Bench size: 67

Sample row:
  [dataset]: virtassist
  [encounter_id]: D2N001
  [dialogue]: [doctor] hi , martha . how are you ?
[patient] i'm doing okay . how are you ?
[doctor] i'm doing okay . so , i know the nurse told you about dax . i'd like to tell dax a little bit about you , okay ?

  [note]: CHIEF COMPLAINT

Annual exam.

HISTORY OF PRESENT ILLNESS

Martha Collins is a 50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension who


In [ ]:
# Cell 14 — ACI-Bench converter
def convert_aci(row, dialogue_col, note_col):
    dialogue_raw = str(row[dialogue_col])
    note = str(row[note_col]) if note_col else ""
    lines = dialogue_raw.strip().split("\n")
    doctor_utts, patient_utts, full_lines = [], [], []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith("[doctor]") or line.startswith("D:") or line.lower().startswith("doctor:"):
            if line.startswith("D:"):
                text = line[2:].strip()
            elif line.startswith("[doctor]"):
                text = line[8:].strip()
            else:
                text = line[7:].strip()
            doctor_utts.append(text)
            full_lines.append(f"[DOCTOR]: {text}")
        elif line.startswith("[patient]") or line.startswith("P:") or line.lower().startswith("patient:"):
            if line.startswith("P:"):
                text = line[2:].strip()
            elif line.startswith("[patient]"):
                text = line[9:].strip()
            else:
                text = line[8:].strip()
            patient_utts.append(text)
            full_lines.append(f"[PATIENT]: {text}")
    return {
        "source": "aci_bench",
        "full_dialogue": "\n".join(full_lines),
        "doctor_utterances": " ".join(doctor_utts),
        "patient_utterances": " ".join(patient_utts),
        "target_note": note,
        "doctor_is_ground_truth": True
    }

# Auto-detect column names
cols = aci_df.columns.tolist()
dialogue_col = next((c for c in cols if any(
    k in c.lower() for k in ["dialogue", "transcript", "conversation"]
)), cols[0])
note_col = next((c for c in cols if any(
    k in c.lower() for k in ["note", "summary", "reference"]
)), None)

print(f"Using dialogue column: {dialogue_col}")
print(f"Using note column: {note_col}")

aci_converted = []
for _, row in aci_df.iterrows():
    try:
        c = convert_aci(row, dialogue_col, note_col)
        if c["doctor_utterances"] and c["patient_utterances"]:
            aci_converted.append(c)
    except:
        continue

print(f"ACI-Bench converted: {len(aci_converted)} examples")

Using dialogue column: dialogue
Using note column: note
ACI-Bench converted: 67 examples


In [ ]:
# Cell 15 — Combine, filter and save unified text dataset
import random, json

all_converted = mts_converted + notechat_converted + aci_converted
random.seed(42)
random.shuffle(all_converted)

print(f"Total before filtering: {len(all_converted)}")

filtered = []
for ex in all_converted:
    if len(ex["doctor_utterances"].split()) < 20:
        continue
    if len(ex["patient_utterances"].split()) < 5:
        continue
    if len(ex["target_note"].strip()) < 30:
        continue
    filtered.append(ex)

print(f"After filtering: {len(filtered)}")
sources = {}
for ex in filtered:
    sources[ex["source"]] = sources.get(ex["source"], 0) + 1
print(f"Breakdown: {sources}")

with open("/kaggle/working/data/training/unified_dataset.json", "w") as f:
    json.dump(filtered, f)
print("Saved unified_dataset.json")

Total before filtering: 3211
After filtering: 2763
Breakdown: {'notechat': 1961, 'mts_dialog': 735, 'aci_bench': 67}
Saved unified_dataset.json


In [ ]:
import os
os.makedirs("/kaggle/working/data/training", exist_ok=True)

with open("/kaggle/working/data/training/unified_dataset.json", "w") as f:
    import json
    json.dump(filtered, f)

print(f"Saved {len(filtered)} examples to unified_dataset.json")

Saved 2763 examples to unified_dataset.json


In [ ]:
# Cell 16 — Download PriMock57 real clinical audio
import subprocess, os

print("Downloading PriMock57 real GP consultation recordings...")
subprocess.run([
    "git", "clone",
    "https://github.com/babylonhealth/primock57.git",
    "/kaggle/working/data/primock57"
], check=True)

# Check what audio files are available
audio_files = []
for root, dirs, files in os.walk("/kaggle/working/data/primock57"):
    for f in files:
        if f.endswith((".wav", ".mp3", ".flac", ".m4a")):
            audio_files.append(os.path.join(root, f))

print(f"\nAudio files found: {len(audio_files)}")
for f in audio_files[:10]:
    print(f"  {f}")

Cloning into '/kaggle/working/data/primock57'...



Audio files found: 114
  /kaggle/working/data/primock57/audio/day2_consultation02_patient.wav
  /kaggle/working/data/primock57/audio/day4_consultation01_patient.wav
  /kaggle/working/data/primock57/audio/day5_consultation07_doctor.wav
  /kaggle/working/data/primock57/audio/day5_consultation11_patient.wav
  /kaggle/working/data/primock57/audio/day1_consultation04_doctor.wav
  /kaggle/working/data/primock57/audio/day5_consultation10_doctor.wav
  /kaggle/working/data/primock57/audio/day2_consultation03_patient.wav
  /kaggle/working/data/primock57/audio/day5_consultation01_doctor.wav
  /kaggle/working/data/primock57/audio/day3_consultation03_doctor.wav
  /kaggle/working/data/primock57/audio/day1_consultation13_patient.wav


Filtering content: 100% (114/114), 1.85 GiB | 70.72 MiB/s, done.


In [ ]:
# Cell 17 — Synthesize audio
from gtts import gTTS
from pydub import AudioSegment
import os, json
from tqdm import tqdm

SYNTH_DIR = "/kaggle/working/data/synthesized_audio"
os.makedirs(SYNTH_DIR, exist_ok=True)

def synthesize_gtts(text, output_path, slow=False):
    try:
        words = text.split()
        if len(words) > 60:
            text = " ".join(words[:60])
        if len(text.strip()) < 3:
            return False
        tts = gTTS(text=text, lang="en", slow=slow)
        mp3_path = output_path.replace(".wav", ".mp3")
        tts.save(mp3_path)
        audio = AudioSegment.from_mp3(mp3_path)
        audio = audio.set_frame_rate(16000).set_channels(1)
        audio.export(output_path, format="wav")
        os.remove(mp3_path)
        return True
    except:
        return False

print("\nSynthesizing audio using ground truth dataset tags...")

synthesis_records = []
classification_stats = {"DOCTOR": 0, "PATIENT": 0}

for idx, ex in enumerate(tqdm(mts_converted[:100])):
    example_dir = f"{SYNTH_DIR}/example_{idx:04d}"
    os.makedirs(example_dir, exist_ok=True)

    lines = ex["full_dialogue"].split("\n")
    doc_files = []
    pat_files = []

    for line_idx, line in enumerate(lines):
        line = line.strip()
        if not line:
            continue

        # Strictly rely on the tags provided in the text
        if line.startswith("[DOCTOR]:"):
            final_role = "DOCTOR"
            text = line[9:].strip()
        elif line.startswith("[PATIENT]:"):
            final_role = "PATIENT"
            text = line[10:].strip()
        else:
            continue

        if len(text) < 5:
            continue

        classification_stats[final_role] = classification_stats.get(final_role, 0) + 1

        filename = f"utt_{line_idx:03d}_{final_role.lower()}.wav"
        output_path = os.path.join(example_dir, filename)

        # Keep the speed trick to differentiate voices
        slow = (final_role == "PATIENT")
        success = synthesize_gtts(text, output_path, slow=slow)

        if success:
            entry = {
                "path": output_path,
                "text": text,
                "speaker": final_role,
                "classification_method": "dataset_ground_truth"
            }
            if final_role == "DOCTOR":
                doc_files.append(entry)
            else:
                pat_files.append(entry)

    if doc_files and pat_files:
        synthesis_records.append({
            "example_id": idx,
            "example_dir": example_dir,
            "doctor_files": doc_files,
            "patient_files": pat_files,
            "target_note": ex["target_note"],
            "source": ex["source"]
        })

print(f"\nSynthesized: {len(synthesis_records)} examples")
print(f"Doctor utterances: {sum(len(r['doctor_files']) for r in synthesis_records)}")
print(f"Patient utterances: {sum(len(r['patient_files']) for r in synthesis_records)}")
print(f"Classification stats (Ground Truth): {classification_stats}")

with open(f"{SYNTH_DIR}/synthesis_records.json", "w") as f:
    json.dump(synthesis_records, f, indent=2)
print("Saved synthesis_records.json")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):



Synthesizing audio using ground truth dataset tags...


100%|██████████| 100/100 [04:59<00:00,  2.99s/it]


Synthesized: 96 examples
Doctor utterances: 398
Patient utterances: 332
Classification stats (Ground Truth): {'DOCTOR': 402, 'PATIENT': 332}
Saved synthesis_records.json


In [ ]:
# Cell 18 — Prepare Whisper fine-tuning dataset
import whisper
import torch
import json
import os

print("Loading Whisper medium...")
device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = whisper.load_model("medium", device=device)
print(f"Whisper loaded on {device}")

# Build Whisper training pairs from synthesized audio
whisper_training_pairs = []

with open("/kaggle/working/data/synthesized_audio/synthesis_records.json") as f:
    synthesis_records = json.load(f)

for record in synthesis_records:
    # Extract Doctor utterances
    for doc_file in record.get("doctor_files", []):
        if os.path.exists(doc_file["path"]):
            whisper_training_pairs.append({
                "audio_path": doc_file["path"],
                "text": doc_file["text"],
                "speaker_role": doc_file.get("speaker", "DOCTOR")
            })

    # Extract Patient utterances
    for pat_file in record.get("patient_files", []):
        if os.path.exists(pat_file["path"]):
            whisper_training_pairs.append({
                "audio_path": pat_file["path"],
                "text": pat_file["text"],
                "speaker_role": pat_file.get("speaker", "PATIENT")
            })

print(f"Whisper training pairs: {len(whisper_training_pairs)}")

# Compute baseline WER on a sample before fine-tuning
print("\nComputing baseline WER on 20 samples...")
from jiwer import wer

baseline_wer_scores = []
for pair in whisper_training_pairs[:20]:
    try:
        result = whisper_model.transcribe(pair["audio_path"], language="en")
        hypothesis = result["text"].strip().lower()
        reference = pair["text"].strip().lower()
        score = wer(reference, hypothesis)
        baseline_wer_scores.append(score)
    except Exception as e:
        continue

if baseline_wer_scores:
    print(f"Baseline WER (before fine-tuning): {sum(baseline_wer_scores)/len(baseline_wer_scores)*100:.1f}%")

Loading Whisper medium...


100%|██████████████████████████████████████| 1.42G/1.42G [00:12<00:00, 118MiB/s]


Whisper loaded on cuda
Whisper training pairs: 730

Computing baseline WER on 20 samples...
Baseline WER (before fine-tuning): 12.6%


In [ ]:
# Cell 19 — Download PriMock57 real clinical audio
import subprocess, os

print("Downloading PriMock57 real GP consultation recordings...")
result = subprocess.run([
    "git", "clone",
    "https://github.com/babylonhealth/primock57.git",
    "/kaggle/working/data/primock57"
], capture_output=True, text=True)

if result.returncode != 0:
    print(f"Git clone error: {result.stderr}")
else:
    print("Download complete")

# Find all audio files
audio_files = []
for root, dirs, files in os.walk("/kaggle/working/data/primock57"):
    for f in files:
        if f.endswith((".wav", ".mp3", ".flac", ".m4a")):
            audio_files.append(os.path.join(root, f))

print(f"Audio files found: {len(audio_files)}")
for f in audio_files[:10]:
    print(f"  {f}")

# Also list all files to understand structure
print("\nAll files in PriMock57:")
for root, dirs, files in os.walk("/kaggle/working/data/primock57"):
    for f in files:
        print(f"  {os.path.join(root, f)}")

Git clone error: fatal: destination path '/kaggle/working/data/primock57' already exists and is not an empty directory.

Audio files found: 114
  /kaggle/working/data/primock57/audio/day2_consultation02_patient.wav
  /kaggle/working/data/primock57/audio/day4_consultation01_patient.wav
  /kaggle/working/data/primock57/audio/day5_consultation07_doctor.wav
  /kaggle/working/data/primock57/audio/day5_consultation11_patient.wav
  /kaggle/working/data/primock57/audio/day1_consultation04_doctor.wav
  /kaggle/working/data/primock57/audio/day5_consultation10_doctor.wav
  /kaggle/working/data/primock57/audio/day2_consultation03_patient.wav
  /kaggle/working/data/primock57/audio/day5_consultation01_doctor.wav
  /kaggle/working/data/primock57/audio/day3_consultation03_doctor.wav
  /kaggle/working/data/primock57/audio/day1_consultation13_patient.wav

All files in PriMock57:
  /kaggle/working/data/primock57/.gitattributes
  /kaggle/working/data/primock57/CONTRIBUTING.md
  /kaggle/working/data/primoc

In [ ]:
# Cell 20 — Whisper fine-tuning on synthesized clinical audio
# Fine-tunes Whisper medium on our synthesized doctor-patient audio
# to improve WER on medical terminology

import torch
import whisper
import json
import os
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

# Load synthesis records
with open("/kaggle/working/data/synthesized_audio/synthesis_records.json") as f:
    synthesis_records = json.load(f)

# Build training pairs
all_pairs = []
for record in synthesis_records:
    for entry in record["doctor_files"] + record["patient_files"]:
        if os.path.exists(entry["path"]) and len(entry["text"].strip()) > 5:
            all_pairs.append({
                "audio_path": entry["path"],
                "text": entry["text"],
                "speaker": entry["speaker"]
            })

print(f"Total fine-tuning pairs: {len(all_pairs)}")

# Split 90/10
split_idx = int(len(all_pairs) * 0.9)
train_pairs = all_pairs[:split_idx]
val_pairs = all_pairs[split_idx:]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)}")


class WhisperDataset(Dataset):
    def __init__(self, pairs, model):
        self.pairs = pairs
        self.model = model

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        try:
            # Load and pad audio to 30 seconds (Whisper standard)
            audio = whisper.load_audio(pair["audio_path"])
            audio = whisper.pad_or_trim(audio)
            mel = whisper.log_mel_spectrogram(audio).to(next(self.model.parameters()).device)
            return mel, pair["text"]
        except Exception as e:
            # Return silent audio on error
            audio = np.zeros(whisper.audio.SAMPLE_RATE * 30, dtype=np.float32)
            audio = whisper.pad_or_trim(audio)
            mel = whisper.log_mel_spectrogram(audio).to(next(self.model.parameters()).device)
            return mel, pair["text"]


def evaluate_wer(model, pairs, n=30):
    from jiwer import wer
    scores = []
    for pair in pairs[:n]:
        try:
            result = model.transcribe(pair["audio_path"], language="en")
            hyp = result["text"].strip().lower()
            ref = pair["text"].strip().lower()
            scores.append(wer(ref, hyp))
        except:
            continue
    return sum(scores) / len(scores) if scores else 1.0


# Fine-tune for 3 epochs
print("\nStarting Whisper fine-tuning...")
WHISPER_SAVE_PATH = "/kaggle/working/models/whisper_finetuned"
os.makedirs(WHISPER_SAVE_PATH, exist_ok=True)

optimizer = AdamW(whisper_model.parameters(), lr=1e-5, weight_decay=0.01)
whisper_model.train()

EPOCHS = 3
BATCH_SIZE = 4

for epoch in range(EPOCHS):
    total_loss = 0
    batch_count = 0

    # Process in batches manually
    for i in tqdm(range(0, len(train_pairs), BATCH_SIZE), desc=f"Epoch {epoch+1}/{EPOCHS}"):
        batch = train_pairs[i:i+BATCH_SIZE]
        batch_loss = 0

        for pair in batch:
            try:
                audio = whisper.load_audio(pair["audio_path"])
                audio = whisper.pad_or_trim(audio)
                mel = whisper.log_mel_spectrogram(audio).to(whisper_model.device).unsqueeze(0)

                # Tokenize text
                tokens = torch.tensor(
                    [whisper_model.tokenizer.encode(pair["text"])]
                ).to(whisper_model.device)

                # Forward pass
                output = whisper_model(mel, tokens[:, :-1])
                loss = torch.nn.functional.cross_entropy(
                    output.view(-1, output.size(-1)),
                    tokens[:, 1:].reshape(-1),
                    ignore_index=-100
                )
                batch_loss += loss
            except:
                continue

        if batch_loss > 0:
            optimizer.zero_grad()
            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(whisper_model.parameters(), 1.0)
            optimizer.step()
            total_loss += batch_loss.item()
            batch_count += 1

    avg_loss = total_loss / max(batch_count, 1)

    # Evaluate WER
    whisper_model.eval()
    val_wer = evaluate_wer(whisper_model, val_pairs)
    whisper_model.train()

    print(f"Epoch {epoch+1}: Loss={avg_loss:.4f} | Val WER={val_wer*100:.1f}%")

# Save fine-tuned Whisper
torch.save(whisper_model.state_dict(), os.path.join(WHISPER_SAVE_PATH, "whisper_clinical.pt"))
print(f"\nWhisper fine-tuned model saved to {WHISPER_SAVE_PATH}")

# Final WER comparison
whisper_model.eval()
final_wer = evaluate_wer(whisper_model, val_pairs, n=50)
print(f"Final WER after fine-tuning: {final_wer*100:.1f}%")

Total fine-tuning pairs: 696
Train: 626 | Val: 70

Starting Whisper fine-tuning...


Epoch 1/3: 100%|██████████| 157/157 [01:10<00:00,  2.22it/s]


Epoch 1: Loss=0.0000 | Val WER=3.3%


Epoch 2/3: 100%|██████████| 157/157 [01:10<00:00,  2.21it/s]


Epoch 2: Loss=0.0000 | Val WER=3.3%


Epoch 3/3: 100%|██████████| 157/157 [01:11<00:00,  2.21it/s]


Epoch 3: Loss=0.0000 | Val WER=3.3%

Whisper fine-tuned model saved to /kaggle/working/models/whisper_finetuned
Final WER after fine-tuning: 4.0%


In [ ]:
# Cell 21 — PriMock57 real audio pipeline validation
# Uses real doctor/patient audio + real transcripts + real clinical notes
# This is our end-to-end test on real data

import os, json, subprocess
import whisper
import torch
from jiwer import wer

PRIMOCK_DIR = "/kaggle/working/data/primock57"
AUDIO_DIR = f"{PRIMOCK_DIR}/audio"
TRANSCRIPT_DIR = f"{PRIMOCK_DIR}/transcripts"
NOTES_DIR = f"{PRIMOCK_DIR}/notes"

# Step 1 — Extract transcripts from TextGrid files using their own script
print("Extracting transcripts from TextGrid files...")
textgrid_script = f"{PRIMOCK_DIR}/scripts/textgrid_to_transcript.py"

# Install required packages for their script
subprocess.run(["pip", "install", "-q", "textgrid"], capture_output=True)

def parse_textgrid(tg_path):
    """
    Parse PriMock57 TextGrid file to extract timestamped words.
    TextGrid format: Praat annotation format with tiers.
    """
    try:
        import textgrid as tg_lib
        tg = tg_lib.TextGrid.fromFile(tg_path)
        words = []
        for tier in tg.tiers:
            for interval in tier:
                if interval.mark and interval.mark.strip():
                    words.append({
                        "start": round(float(interval.minTime), 3),
                        "end": round(float(interval.maxTime), 3),
                        "word": interval.mark.strip()
                    })
        return " ".join([w["word"] for w in words if w["word"] not in ["", "sp", "<breath>", "<noise>"]])
    except Exception as e:
        return ""


# Step 2 — Build consultation pairs
# Each consultation has: doctor.wav + patient.wav + doctor.TextGrid + patient.TextGrid + notes/day_consult.json
print("Building consultation pairs...")

def get_consultation_id(filename):
    # e.g. day4_consultation02_doctor.wav → day4_consultation02
    parts = os.path.splitext(filename)[0].rsplit("_", 1)
    return parts[0]

# Find all unique consultation IDs
audio_files = [f for f in os.listdir(AUDIO_DIR) if f.endswith(".wav")]
consultation_ids = set()
for f in audio_files:
    cid = get_consultation_id(f)
    consultation_ids.add(cid)

print(f"Found {len(consultation_ids)} unique consultations")

# Build full pairs
consultations = []
for cid in sorted(consultation_ids):
    doc_audio = os.path.join(AUDIO_DIR, f"{cid}_doctor.wav")
    pat_audio = os.path.join(AUDIO_DIR, f"{cid}_patient.wav")
    doc_tg = os.path.join(TRANSCRIPT_DIR, f"{cid}_doctor.TextGrid")
    pat_tg = os.path.join(TRANSCRIPT_DIR, f"{cid}_patient.TextGrid")
    note_path = os.path.join(NOTES_DIR, f"{cid}.json")

    has_doctor_audio = os.path.exists(doc_audio)
    has_patient_audio = os.path.exists(pat_audio)
    has_note = os.path.exists(note_path)

    if has_doctor_audio and has_patient_audio:
        consultation = {
            "id": cid,
            "doctor_audio": doc_audio,
            "patient_audio": pat_audio,
            "doctor_textgrid": doc_tg if os.path.exists(doc_tg) else None,
            "patient_textgrid": pat_tg if os.path.exists(pat_tg) else None,
            "note_path": note_path if has_note else None,
        }

        # Load reference note if available
        if has_note:
            with open(note_path) as f:
                consultation["reference_note"] = json.load(f)

        consultations.append(consultation)

print(f"Complete consultation pairs: {len(consultations)}")
print(f"With reference notes: {sum(1 for c in consultations if c.get('reference_note'))}")

Extracting transcripts from TextGrid files...
Building consultation pairs...
Found 57 unique consultations
Complete consultation pairs: 57
With reference notes: 57


In [ ]:
# Cell 21b — Run Whisper on real PriMock57 audio and compute WER

print("Loading Whisper medium for real audio transcription...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load fine-tuned whisper if available, else base medium
whisper_path = "/kaggle/working/models/whisper_finetuned/whisper_clinical.pt"
if os.path.exists(whisper_path):
    whisper_model = whisper.load_model("medium", device=device)
    whisper_model.load_state_dict(torch.load(whisper_path, map_location=device))
    print("Loaded fine-tuned Whisper")
else:
    whisper_model = whisper.load_model("medium", device=device)
    print("Loaded base Whisper medium (fine-tuned model not found)")

def transcribe_audio(audio_path):
    """Transcribe a single audio file using Whisper."""
    try:
        result = whisper_model.transcribe(
            audio_path,
            language="en",
            word_timestamps=True,
            condition_on_previous_text=True
        )
        return result["text"].strip(), result["segments"]
    except Exception as e:
        return "", []


# Process first 10 consultations
print(f"\nProcessing {min(10, len(consultations))} real consultations...\n")
wer_scores_doctor = []
wer_scores_patient = []
pipeline_results = []

for consult in consultations[:10]:
    cid = consult["id"]
    print(f"Processing: {cid}")

    result = {
        "id": cid,
        "doctor_transcript": "",
        "patient_transcript": "",
        "reference_transcript_doctor": "",
        "reference_transcript_patient": "",
        "wer_doctor": None,
        "wer_patient": None,
        "full_dialogue": "",
        "reference_note": consult.get("reference_note", {})
    }

    # Transcribe doctor audio
    doc_text, _ = transcribe_audio(consult["doctor_audio"])
    result["doctor_transcript"] = doc_text

    # Transcribe patient audio
    pat_text, _ = transcribe_audio(consult["patient_audio"])
    result["patient_transcript"] = pat_text

    # Get reference transcripts from TextGrid
    if consult["doctor_textgrid"]:
        ref_doc = parse_textgrid(consult["doctor_textgrid"])
        result["reference_transcript_doctor"] = ref_doc
        if ref_doc and doc_text:
            score = wer(ref_doc.lower(), doc_text.lower())
            result["wer_doctor"] = round(score, 4)
            wer_scores_doctor.append(score)

    if consult["patient_textgrid"]:
        ref_pat = parse_textgrid(consult["patient_textgrid"])
        result["reference_transcript_patient"] = ref_pat
        if ref_pat and pat_text:
            score = wer(ref_pat.lower(), pat_text.lower())
            result["wer_patient"] = round(score, 4)
            wer_scores_patient.append(score)

    # Build interleaved dialogue with authority labels
    # Since filenames tell us who is speaking, labels are 100% accurate
    full_dialogue = f"[DOCTOR]: {doc_text}\n[PATIENT]: {pat_text}"
    result["full_dialogue"] = full_dialogue

    print(f"  Doctor WER: {result['wer_doctor']}")
    print(f"  Patient WER: {result['wer_patient']}")
    print(f"  Doctor transcript (first 100): {doc_text[:100]}")
    print(f"  Patient transcript (first 100): {pat_text[:100]}")

    pipeline_results.append(result)

# Summary WER
if wer_scores_doctor:
    print(f"\nAverage Doctor WER on real audio: {sum(wer_scores_doctor)/len(wer_scores_doctor)*100:.1f}%")
if wer_scores_patient:
    print(f"Average Patient WER on real audio: {sum(wer_scores_patient)/len(wer_scores_patient)*100:.1f}%")

# Save results
with open("/kaggle/working/data/diarized/primock57_pipeline_results.json", "w") as f:
    json.dump(pipeline_results, f, indent=2)
print(f"\nResults saved for {len(pipeline_results)} consultations")

Loading Whisper medium for real audio transcription...
Loaded fine-tuned Whisper

Processing 10 real consultations...

Processing: day1_consultation01
  Doctor WER: 0.3045
  Patient WER: 0.3071
  Doctor transcript (first 100): Hello. Hi. Can we stop? Yeah, okay. Hello. Good morning. So how can I help you this morning? Sorry t
  Patient transcript (first 100): Hello, how are you? Hi. I've just had some diarrhea for the last three days, and it's been affecting
Processing: day1_consultation02
  Doctor WER: 0.2759
  Patient WER: 0.3133
  Doctor transcript (first 100): Hello? Yes, I think it's a bit better. It's a bit, it's not very clear, but let's continue anyway. O
  Patient transcript (first 100): Hello, can you hear me well? OK. Yes, so it's been a few days now. I have like a sore and a red skin
Processing: day1_consultation03
  Doctor WER: 0.2791
  Patient WER: 0.308
  Doctor transcript (first 100): Hello. Hello there. It's Dr. C there. How can I help you this afternoon? Sorry to hear

In [ ]:
import os
import json

os.makedirs("/kaggle/working/data/diarized", exist_ok=True)

with open("/kaggle/working/data/diarized/primock57_pipeline_results.json", "w") as f:
    json.dump(pipeline_results, f, indent=2)

print(f"✅ Data successfully rescued and saved for {len(pipeline_results)} consultations!")

✅ Data successfully rescued and saved for 10 consultations!


In [ ]:
# ============================================================
# CELL 22 — Free only what we no longer need
# ============================================================
import torch, gc, ctypes

print("Freeing ASR/diarization models only...")

for var_name in ["whisper_model", "diarization_pipeline", "trainer"]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

try:
    ctypes.CDLL("libc.so.6").malloc_trim(0)
except:
    pass

print("GPU memory after cleanup:")
for i in range(torch.cuda.device_count()):
    total    = torch.cuda.get_device_properties(i).total_memory
    reserved = torch.cuda.memory_reserved(i)
    print(f"  GPU {i}: {(total-reserved)/1e9:.2f} GB free")

print(f"\nfiltered: {len(filtered)} examples — OK")
print(f"pipeline_results: {len(pipeline_results)} consultations — OK")
print(f"retrieve: {'OK' if callable(retrieve) else 'MISSING'}")
print(f"embedder: {'OK' if embedder is not None else 'MISSING'}")

Freeing ASR/diarization models only...
GPU memory after cleanup:
  GPU 0: 12.56 GB free
  GPU 1: 15.64 GB free

filtered: 2763 examples — OK
pipeline_results: 10 consultations — OK
retrieve: OK
embedder: OK


In [ ]:
# ============================================================
# CELL 23 — Load Llama-3-8B (4-bit Quantized for T4 GPUs)
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

# We use the ungated NousResearch version of Llama-3-8B-Instruct
MODEL_NAME = "NousResearch/Meta-Llama-3-8B-Instruct"

print(f"Loading {MODEL_NAME} tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Llama-3 doesn't have a default pad token, so we assign one for training
tokenizer.pad_token = tokenizer.eos_token

print("Configuring 4-bit Quantization (QLoRA)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading model across GPUs natively...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

# Enable gradient checkpointing to save VRAM
model.gradient_checkpointing_enable()

# Configure LoRA for Causal LM training
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\nGPU 0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB used")
print(f"GPU 1: {torch.cuda.memory_allocated(1)/1e9:.2f} GB used")
print("Llama-3 is ready for fine-tuning!")

Loading NousResearch/Meta-Llama-3-8B-Instruct tokenizer...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848

GPU 0: 4.68 GB used
GPU 1: 4.11 GB used
Llama-3 is ready for fine-tuning!


In [ ]:
# ============================================================
# CELL 24 — Build Closed-Loop Ecosystem Training Data (RAFT)
# ============================================================
import json, random
from datasets import load_dataset
from tqdm import tqdm

print("Fetching REAL training data...")
try:

    with open("/kaggle/working/data/training/unified_dataset.json", "r") as f:
        train_data = json.load(f)

    # Shuffle so we get a random mix of NoteChat, MTS-Dialog, and ACI-Bench
    random.seed(42)
    random.shuffle(train_data)
    print(f"Loaded {len(train_data)} custom examples from your ecosystem!")
except Exception as e:
    print(f"Error loading dataset: {e}")
    train_data = []

llama_examples = []
print("Fusing Transcripts with Chroma DB PDFs to create RAFT Training Data...")

MAX_EXAMPLES = 300

for row in tqdm(list(train_data)[:MAX_EXAMPLES]):
    dialogue = row.get('full_dialogue', row.get('dialogue', row.get('text', '')))
    reference_note = row.get('target_note', row.get('summary', row.get('note', '')))

    if not dialogue or len(dialogue) < 50:
        continue

    if not reference_note:
        reference_note = "Patient presented for consultation. Discussed ongoing symptoms and plan."

    # 1. Simulate the RAG lookup during training!
    # We grab the first 50 words of the dialogue to find relevant PDF rules
    search_query = " ".join(dialogue.split()[:50])

    try:
        rag_knowledge = laos_retrieval_pipeline(search_query, domain="general", top_k_initial=3, top_k_final=1)
    except:
        rag_knowledge = "No specific guidelines retrieved."

    target_json = json.dumps({"clinical_note": reference_note}, indent=2)

    full_text = (
        f"### Expertise:\n"
        f"You are a professional medical specialist.\n\n"
        f"### Task-specific Instruction:\n"
        f"Create a diagnostic document for the patient based on the provided dialogue.\n\n"
        f"### Instructions:\n"
        f"- Use JSON format for data.\n"
        f"- Use plain, professional language.\n"
        f"- Avoid redundancy in examinations.\n"
        f"- Output MUST adhere to the medical facts retrieved below.\n\n"
        f"### Common Terms & Guidelines (Retrieved Knowledge):\n"
        f"{rag_knowledge if rag_knowledge else 'None found. Rely strictly on dialogue.'}\n\n"
        f"### Input Materials:\n"
        f"{dialogue}\n\n"
        f"### Output:\n"
        f"{target_json}"
    )

    llama_examples.append({"text": full_text})

print(f"\nBuilt {len(llama_examples)} Ecosystem-reliant Llama-3 training examples!")

random.seed(42)
random.shuffle(llama_examples)
split = int(len(llama_examples) * 0.9)
train_ex = llama_examples[:split]
val_ex = llama_examples[split:]
print(f"Train: {len(train_ex)} | Val: {len(val_ex)}")

Fetching REAL training data...
Loaded 2763 custom examples from your ecosystem!
Fusing Transcripts with Chroma DB PDFs to create RAFT Training Data...



100%|██████████| 300/300 [00:00<00:00, 21541.66it/s]


Built 300 Ecosystem-reliant Llama-3 training examples!
Train: 270 | Val: 30


In [ ]:
# ============================================================
# CELL 25 — Convert to HuggingFace Dataset & Sanity Check
# ============================================================
from datasets import Dataset

print("Converting custom RAFT examples into HuggingFace Datasets...")

train_dataset = Dataset.from_list(train_ex)
val_dataset = Dataset.from_list(val_ex)

print(f"✅ Train Dataset ready: {len(train_dataset)} rows")
print(f"✅ Validation Dataset ready: {len(val_dataset)} rows\n")

print("="*80)
print("Here is exactly what Llama-3 will see during training:")
print("="*80)

# Print the very first training example to verify the strict LAOS formatting
print(train_dataset[0]["text"])
print("\n" + "="*80)
print("Ready for LoRA Fine-Tuning in Cell 26!")

Converting custom RAFT examples into HuggingFace Datasets...
✅ Train Dataset ready: 270 rows
✅ Validation Dataset ready: 30 rows

Here is exactly what Llama-3 will see during training:
### Expertise:
You are a professional medical specialist.

### Task-specific Instruction:
Create a diagnostic document for the patient based on the provided dialogue.

### Instructions:
- Use JSON format for data.
- Use plain, professional language.
- Avoid redundancy in examinations.
- Output MUST adhere to the medical facts retrieved below.

### Common Terms & Guidelines (Retrieved Knowledge):
No specific guidelines retrieved.

### Input Materials:
[DOCTOR]: Hello, I will ask you a few basic questions, okay?
[PATIENT]: Okay.
[DOCTOR]: What is your living status? Do you live alone or with family?
[PATIENT]: I am with my husband.
[DOCTOR]: What is your education level?
[PATIENT]: I cleared my twelfth grade but never joined college after that.
[DOCTOR]: Um, did you ever take any kind of illicit drugs?
[PA

In [ ]:
# ============================================================
# CELL 26 — RAFT LoRA Fine-Tuning (The Medical Residency)
# ============================================================
import torch
import os
import inspect
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

print("Configuring LoRA Adapter for the Closed-Loop Ecosystem...")

model.config.use_cache = False

# CRITICAL: Guarantee gradients flow even if Kaggle re-runs the cell
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

# Check if the model is already a PeftModel from a previous Kaggle run
is_already_peft = isinstance(model, PeftModel) or hasattr(model, "peft_config")

if not is_already_peft:
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# Dynamically build training arguments
args_kwargs = {
    "output_dir": "/kaggle/working/models/checkpoints/llama3_raft_v1",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "optim": "paged_adamw_32bit", # PERMANENT FIX: Stable 32-bit optimizer
    "save_steps": 50,
    "logging_steps": 10,
    "learning_rate": 2e-4,
    "fp16": False, # PERMANENT FIX: Disables the AMP scaler that caused the AssertionError
    "max_grad_norm": 0.3,
    "num_train_epochs": 3,
    "warmup_steps": 10,
    "group_by_length": True,
    "lr_scheduler_type": "constant",
    "report_to": "none"
}

if "dataset_text_field" in inspect.signature(SFTConfig.__init__).parameters:
    args_kwargs["dataset_text_field"] = "text"

if "max_seq_length" in inspect.signature(SFTConfig.__init__).parameters:
    args_kwargs["max_seq_length"] = 2048

training_args = SFTConfig(**args_kwargs)
tokenizer.model_max_length = 2048

# Dynamically build Trainer arguments
trainer_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "eval_dataset": val_dataset,
    "args": training_args,
}

if "max_seq_length" in inspect.signature(SFTTrainer.__init__).parameters and "max_seq_length" not in args_kwargs:
    trainer_kwargs["max_seq_length"] = 2048

# Pass peft_config only if the model isn't already wrapped
if not is_already_peft:
    trainer_kwargs["peft_config"] = peft_config
else:
    print("Model already wrapped with PEFT. Skipping peft_config in Trainer to prevent crash.")

# The Hugging Face version fix for tokenizer/processing_class
sft_params = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sft_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_params:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    trainer_kwargs["processing_class"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("="*80)
print("🚀 INITIATING RAFT TRAINING...")
print("Teaching Llama-3 to strictly obey your proprietary Chroma DB guidelines.")
print("="*80)

trainer.train()

ADAPTER_SAVE_PATH = "/kaggle/working/models/lora_adapters/llama3_clinical_raft_v1"
os.makedirs(ADAPTER_SAVE_PATH, exist_ok=True)

trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print("\n" + "="*80)
print(f"🎉 SUCCESS! Proprietary RAFT Ecosystem Model saved to: {ADAPTER_SAVE_PATH}")
print("="*80)

Configuring LoRA Adapter for the Closed-Loop Ecosystem...
Model already wrapped with PEFT. Skipping peft_config in Trainer to prevent crash.


Adding EOS to train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/270 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

🚀 INITIATING RAFT TRAINING...
Teaching Llama-3 to strictly obey your proprietary Chroma DB guidelines.


Step,Training Loss
10,1.644555
20,2.125467
30,1.727897
40,1.856222
50,2.075698
60,1.692623
70,2.186072
80,1.698600
90,2.047478
100,2.078641



🎉 SUCCESS! Proprietary RAFT Ecosystem Model saved to: /kaggle/working/models/lora_adapters/llama3_clinical_raft_v1


In [ ]:
# ============================================================
# CELL 27 — Ecosystem Inference (Testing the Lobotomized AI)
# ============================================================
import json
import torch
from tqdm import tqdm

print("Switching model to Evaluation Mode...")
trainer.model.eval() # Freezes the weights for testing

# Test on ALL 30 validation examples for a highly accurate evaluation!
test_cases = val_ex
results = []

print(f"Generating clinical notes for {len(test_cases)} validation cases...")

for i, case in enumerate(tqdm(test_cases)):

    full_text = case["text"]

    try:
        prompt_part = full_text.split("### Output:\n")[0] + "### Output:\n"
        reference_json = full_text.split("### Output:\n")[1]
    except IndexError:
        continue

    inputs = tokenizer(prompt_part, return_tensors="pt").to(trainer.model.device)

    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=600,
            pad_token_id=tokenizer.eos_token_id,
            temperature=0.1,
            repetition_penalty=1.1,
            do_sample=True,
        )

    input_length = inputs["input_ids"].shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

    results.append({
        "prompt": prompt_part,
        "reference_note": reference_json,
        "generated_note": generated_text
    })

OUTPUT_SAVE_PATH = "/kaggle/working/primock57_generated_notes.json"
with open(OUTPUT_SAVE_PATH, "w") as f:
    json.dump(results, f, indent=4)

print("\n" + "="*80)
print(f"✅ Generated {len(results)} ecosystem-reliant notes!")
print(f"✅ Saved to {OUTPUT_SAVE_PATH}")
print("="*80)

Switching model to Evaluation Mode...
Generating clinical notes for 30 validation cases...



100%|██████████| 30/30 [25:13<00:00, 50.45s/it]


✅ Generated 30 ecosystem-reliant notes!
✅ Saved to /kaggle/working/primock57_generated_notes.json


In [ ]:
# ============================================================
# CELL 28 — Review All 30 Generated Notes
# ============================================================
import json

OUTPUT_SAVE_PATH = "/kaggle/working/primock57_generated_notes.json"

print(f"Loading generated notes from {OUTPUT_SAVE_PATH}...")
with open(OUTPUT_SAVE_PATH, "r") as f:
    results = json.load(f)

print(f"Successfully loaded {len(results)} cases. Displaying comparison...\n")

for i, case in enumerate(results):
    print("="*80)
    print(f"🩺 CASE {i+1} / {len(results)}")
    print("="*80)

    print("\n--- GENERATED NOTE (FROM ECOSYSTEM) ---")
    print(case["generated_note"].strip())

    print("\n--- 🧑‍⚕️ GROUND TRUTH DOCTOR NOTE (REFERENCE) ---")
    print(case["reference_note"].strip())
    print("\n")

Loading generated notes from /kaggle/working/primock57_generated_notes.json...
Successfully loaded 30 cases. Displaying comparison...

🩺 CASE 1 / 30

--- GENERATED NOTE (FROM ECOSYSTEM) ---
```
{
    "Patient ID": "123456",
    "Date of Visit": "2023-02-20",
    "Chief Complaint": "Recurring pelvic pain and dysmenorrhea",
    "Medical History":
        {
            "Endometriosis": "Diagnosed 9 years ago, treated with laparoscopy and GnRH analogues",
            "Hepatic Adenoma": "Found during OCP treatment"
        },
    "Physical Examination":
        {
            "Pelvic Examination": "Normal",
            "Abdominal Examination": "Normal",
            "Right Shoulder Pain": "Present"
        },
    "Diagnostic Impressions":
        [
            "Chronic pelvic pain and dysmenorrhea likely related to persistent endometriosis",
            "Dyspareunia possibly related to residual endometriotic lesions or adhesions"
        ],
    "Treatment Plan":
        {
            "Follow-

In [ ]:
# ============================================================
# CELL 29
# ============================================================
import json, re, warnings
import evaluate
warnings.filterwarnings("ignore")

OUTPUT_SAVE_PATH = "/kaggle/working/primock57_generated_notes.json"

print("Loading evaluation metrics (ROUGE & BERTScore)...")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

with open(OUTPUT_SAVE_PATH, "r") as f:
    results = json.load(f)

def flatten_to_string(obj):
    """Recursively flattens lists and dictionaries into a single space-separated string."""
    vals = []
    if isinstance(obj, dict):
        for v in obj.values(): vals.extend(flatten_to_string(v).split())
    elif isinstance(obj, list):
        for v in obj: vals.extend(flatten_to_string(v).split())
    else:
        vals.append(str(obj))
    return " ".join(vals)

def extract_text_from_llm_output(llm_string):
    """Strips the markdown and flattens the JSON into readable text for scoring."""
    match = re.search(r'`{3}(?:json)?(.*?)`{3}', llm_string, re.DOTALL)
    raw_json = match.group(1).strip() if match else llm_string

    try:
        data = json.loads(raw_json)
        return flatten_to_string(data)
    except:
        # Fallback if it's not perfect JSON
        return llm_string.replace("\n", " ")

predictions = []
references = []

for case in results:
    pred_text = extract_text_from_llm_output(case["generated_note"])
    ref_raw = case.get("reference_note", "")

    # Safely convert the reference note to a string if it happens to be a dictionary/list
    if isinstance(ref_raw, (dict, list)):
        ref_text = flatten_to_string(ref_raw)
    else:
        ref_text = str(ref_raw)

    # Only evaluate if a reference note actually exists
    if ref_text and len(ref_text.strip()) > 5:
        predictions.append(pred_text)
        references.append(ref_text)

if len(predictions) > 0:
    print(f"\nScoring {len(predictions)} notes...")

    # 1. ROUGE Score (Word overlap)
    rouge_results = rouge.compute(predictions=predictions, references=references)
    print("\n" + "="*30)
    print("🏆 ROUGE SCORES (Lexical Match)")
    print("="*30)
    print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

    # 2. BERTScore (Semantic meaning - takes a minute to download the model)
    print("\nCalculating BERTScore (Semantic Match)... this might take a moment.")
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")
    print("\n" + "="*30)
    print("🧠 BERTSCORE (Semantic Meaning)")
    print("="*30)
    print(f"Precision: {sum(bert_results['precision']) / len(bert_results['precision']):.4f}")
    print(f"Recall:    {sum(bert_results['recall']) / len(bert_results['recall']):.4f}")
    print(f"F1-Score:  {sum(bert_results['f1']) / len(bert_results['f1']):.4f}")
else:
    print("No reference notes found to compare against! (Make sure your PriMock57 dataset has 'reference_note' fields)")

Loading evaluation metrics (ROUGE & BERTScore)...



Scoring 30 notes...

🏆 ROUGE SCORES (Lexical Match)
ROUGE-1: 0.3586
ROUGE-2: 0.1933
ROUGE-L: 0.2771

Calculating BERTScore (Semantic Match)... this might take a moment.


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🧠 BERTSCORE (Semantic Meaning)
Precision: 0.8503
Recall:    0.8184
F1-Score:  0.8335


In [ ]:
# ============================================================
# CELL 30 — Implementing the LAOS Research Architecture
# ============================================================
import torch
import subprocess
import json
import os
import shutil
import time
import gc

# ⚠️ FLUSH DEAD GPU MEMORY BEFORE STARTING
torch.cuda.empty_cache()
gc.collect()

print("Installing required libraries...")
# ⚠️ WE ARE USING SENTENCE-TRANSFORMERS NOW, NOT FLAGEMBEDDING
subprocess.run(["pip", "install", "-q", "sentence-transformers", "textgrid"], capture_output=True)

from sentence_transformers import CrossEncoder
import textgrid as tg_lib

print("Loading bge-reranker-base (Forcing onto CPU to prevent GPU deadlocks)...")
# ⚠️ FORCING DEVICE='CPU' TO STOP THE FREEZE
reranker = CrossEncoder('BAAI/bge-reranker-base', device='cpu', max_length=512)

def laos_retrieval_pipeline(query, domain, top_k_initial=5, top_k_final=2):
    if not collections:
        return ""

    if domain not in collections:
        domain = list(collections.keys())[0]

    collection = collections[domain]

    query_embedding = embedder.encode(f"Represent this clinical query: {query}", normalize_embeddings=True).tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=top_k_initial, include=["documents", "metadatas"])

    if not results["documents"] or not results["documents"][0]:
        return ""

    retrieved_docs = results["documents"][0]

    # Pair query with each document
    pairs = [[query, doc] for doc in retrieved_docs]

    # Use CrossEncoder's predict instead of FlagEmbedding's dataloader
    scores = reranker.predict(pairs).tolist()

    scored_docs = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)
    final_docs = [doc for score, doc in scored_docs[:top_k_final]]
    return "\n\n".join(final_docs)


def generate_laos_report(dialogue, domain="general"):
    # Shrink dialogue to 250 words to guarantee no GPU VRAM overloads
    truncated_dialogue = " ".join(dialogue.split()[:250])
    search_query = " ".join(truncated_dialogue.split()[:50])

    rag_knowledge = laos_retrieval_pipeline(search_query, domain)

    # Cap the retrieved knowledge
    if len(rag_knowledge) > 1000:
        rag_knowledge = rag_knowledge[:1000] + "... [TRUNCATED FOR MEMORY]"

    prompt = (
        f"### Expertise:\n"
        f"You are a professional medical specialist.\n\n"

        f"### Task-specific Instruction:\n"
        f"Create a diagnostic document for the patient based on the provided dialogue.\n\n"

        f"### Instructions:\n"
        f"- Use JSON format for data.\n"
        f"- Use plain, professional language.\n"
        f"- Avoid redundancy in examinations.\n"
        f"- Output MUST adhere to the medical facts retrieved below.\n\n"

        f"### Common Terms & Guidelines (Retrieved Knowledge):\n"
        f"{rag_knowledge if rag_knowledge else 'None found. Rely strictly on dialogue.'}\n\n"

        f"### Input Materials:\n"
        f"{truncated_dialogue}\n\n"

        f"### Output:\n"
    )

    # Use `trainer.model` to prevent PEFT deadlocks
    active_model = trainer.model if 'trainer' in globals() else model

    inputs = tokenizer(prompt, return_tensors="pt").to(active_model.device)

    # Use inference_mode (faster and safely disables memory gradients)
    with torch.inference_mode():
        outputs = active_model.generate(
            **inputs,
            max_new_tokens=400,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
            temperature=0.1,
            do_sample=True,
        )

    input_length = inputs["input_ids"].shape[1]
    generated = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return generated

# --- RECONSTRUCT TEXTGRIDS FROM CELL 19/21 ---
primock_data = []
PRIMOCK_DIR = "/kaggle/working/data/primock57"
TRANSCRIPT_DIR = os.path.join(PRIMOCK_DIR, "transcripts")
NOTES_DIR = os.path.join(PRIMOCK_DIR, "notes")

print("\nParsing PriMock57 TextGrid Transcripts...")

if os.path.exists(TRANSCRIPT_DIR) and os.path.exists(NOTES_DIR):
    cids = set([f.rsplit("_", 1)[0] for f in os.listdir(TRANSCRIPT_DIR) if f.endswith(".TextGrid")])

    for cid in sorted(cids):
        doc_tg = os.path.join(TRANSCRIPT_DIR, f"{cid}_doctor.TextGrid")
        pat_tg = os.path.join(TRANSCRIPT_DIR, f"{cid}_patient.TextGrid")
        note_path = os.path.join(NOTES_DIR, f"{cid}.json")

        if not os.path.exists(note_path):
            continue

        with open(note_path, "r") as f:
            note_json = json.load(f)
            reference = json.dumps(note_json)

        utterances = []
        def process_tg(path, speaker):
            if not os.path.exists(path): return
            try:
                tg = tg_lib.TextGrid.fromFile(path)
                for tier in tg.tiers:
                    for interval in tier:
                        if interval.mark and interval.mark.strip() not in ["", "sp", "<breath>", "<noise>"]:
                            utterances.append({
                                "speaker": speaker,
                                "start": float(interval.minTime),
                                "word": interval.mark.strip()
                            })
            except Exception:
                pass

        process_tg(doc_tg, "Doctor")
        process_tg(pat_tg, "Patient")

        utterances.sort(key=lambda x: x["start"])

        dialogue_lines = []
        current_speaker = None
        current_sentence = []
        for u in utterances:
            if u["speaker"] != current_speaker:
                if current_speaker is not None:
                    dialogue_lines.append(f"{current_speaker}: {' '.join(current_sentence)}")
                current_speaker = u["speaker"]
                current_sentence = [u["word"]]
            else:
                current_sentence.append(u["word"])
        if current_speaker is not None:
            dialogue_lines.append(f"{current_speaker}: {' '.join(current_sentence)}")

        dialogue = "\n".join(dialogue_lines)

        if dialogue and reference:
            primock_data.append({"dialogue": dialogue, "reference_note": reference})

print(f"✅ Successfully reconstructed {len(primock_data)} real PriMock57 cases!")

print("\nRunning LAOS-style RAG inference on a PriMock57 transcript...")
if primock_data:
    final_note = generate_laos_report(primock_data[0]["dialogue"], domain="general")

    print("="*60)
    print("📝 THE LAOS-ARCHITECTURE GENERATED NOTE:")
    print("="*60)
    print(final_note)
else:
    print("❌ Could not find transcripts in /kaggle/working/data/primock57.")

Installing required libraries...
Loading bge-reranker-base (Forcing onto CPU to prevent GPU deadlocks)...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


README.md: 0.00B [00:00, ?B/s]


Parsing PriMock57 TextGrid Transcripts...
✅ Successfully reconstructed 57 real PriMock57 cases!

Running LAOS-style RAG inference on a PriMock57 transcript...
📝 THE LAOS-ARCHITECTURE GENERATED NOTE:
```
{
  "Patient Information": {
    "Name": "",
    "Age": "",
    "Complaints": [
      {
        "Symptoms": ["Diarrhea", "Loose and Watery Stool", "Pain in Lower Abdomen"],
        "Duration": "Last three days",
        "Frequency": "Six to seven times a day"
      }
    ]
  },
  "Examination Findings": {
    "Physical Examination": {
      "Abdominal Examination": {
        "Tenderness": "Present on the left side",
        "Guarding": "Absent"
      }
    }
  },
  "Assessment and Plan":
  {
    "Assessment": "Acute gastroenteritis with possible inflammatory component",
    "Plan": "Prescription of anti-diarrheal medication and probiotics; follow-up appointment scheduled for [Date]"
  }
}
```


### Additional Notes:
Please ensure that the output adheres to the guidelines provided above

In [ ]:
import json, re, warnings
import evaluate
from tqdm import tqdm
warnings.filterwarnings("ignore")

print(f"🚀 Unleashing LAOS Pipeline on {len(primock_data)} PriMock57 cases...")
laos_results = []

for i, row in enumerate(tqdm(primock_data)):
    dialogue = row.get("dialogue", "")
    reference = row.get("reference_note", "")

    if not dialogue or not reference:
        continue

    try:
        generated_note = generate_laos_report(dialogue, domain="general")
        laos_results.append({
            "prompt": dialogue,
            "reference_note": reference,
            "generated_note": generated_note
        })
    except Exception as e:
        print(f"Skipping case {i} due to error: {e}")

LAOS_OUTPUT_PATH = "/kaggle/working/laos_primock_results.json"
with open(LAOS_OUTPUT_PATH, "w") as f:
    json.dump(laos_results, f, indent=4)

print(f"\n✅ Saved {len(laos_results)} LAOS-generated notes to {LAOS_OUTPUT_PATH}")

print("\nCalculating ROUGE & BERTScore for the final LAOS Pipeline...")

def extract_text_from_llm_output_safe(llm_string):
    match = re.search(r'`{3}(?:json)?(.*?)`{3}', llm_string, re.DOTALL)
    raw_json = match.group(1).strip() if match else llm_string
    try:
        data = json.loads(raw_json)
        vals = []
        def _flatten(obj):
            if isinstance(obj, dict):
                for v in obj.values(): _flatten(v)
            elif isinstance(obj, list):
                for v in obj: _flatten(v)
            else:
                vals.append(str(obj))
        _flatten(data)
        return " ".join(vals)
    except:
        return llm_string.replace("\n", " ")

predictions = []
references = []

for case in laos_results:
    pred_text = extract_text_from_llm_output_safe(case["generated_note"])
    ref_text = str(case["reference_note"])
    predictions.append(pred_text)
    references.append(ref_text)

if len(predictions) > 0:
    rouge = evaluate.load("rouge")
    rouge_results = rouge.compute(predictions=predictions, references=references)

    print("\n" + "="*40)
    print("🏆 LAOS PIPELINE ROUGE SCORES")
    print("="*40)
    print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
    print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
    print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")

    bertscore = evaluate.load("bertscore")
    bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")

    print("\n" + "="*40)
    print("🧠 LAOS PIPELINE BERTSCORE")
    print("="*40)
    print(f"Precision: {sum(bert_results['precision']) / len(bert_results['precision']):.4f}")
    print(f"Recall:    {sum(bert_results['recall']) / len(bert_results['recall']):.4f}")
    print(f"F1-Score:  {sum(bert_results['f1']) / len(bert_results['f1']):.4f}")
else:
    print("No valid cases were generated to score.")

🚀 Unleashing LAOS Pipeline on 57 PriMock57 cases...


100%|██████████| 57/57 [37:54<00:00, 39.91s/it]



✅ Saved 57 LAOS-generated notes to /kaggle/working/laos_primock_results.json

Calculating ROUGE & BERTScore for the final LAOS Pipeline...

🏆 LAOS PIPELINE ROUGE SCORES
ROUGE-1: 0.1566
ROUGE-2: 0.0242
ROUGE-L: 0.0970


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🧠 LAOS PIPELINE BERTSCORE
Precision: 0.8184
Recall:    0.7750
F1-Score:  0.7960


In [ ]:
import json

# Load the saved results from the evaluation cell
RESULTS_PATH = "/kaggle/working/laos_primock_results.json"

try:
    with open(RESULTS_PATH, "r") as f:
        results = json.load(f)

    print(f"✅ Successfully loaded {len(results)} cases for review.\n")
    print("="*80)
    print(" 🏥 LAOS PIPELINE: GROUND TRUTH VS. PREDICTED OUTPUTS")
    print("="*80)

    for i, case in enumerate(results):
        print(f"\n\n{'='*80}")
        print(f"🩺 CASE {i+1} / {len(results)}")
        print(f"{'='*80}")

        # 1. Print Ground Truth (What the real doctor wrote)
        print("\n🧑‍⚕️ GROUND TRUTH (Real Doctor's Note):")
        print("-" * 50)
        # Format if it's a dict, otherwise print raw string
        if isinstance(case["reference_note"], dict):
            print(json.dumps(case["reference_note"], indent=2))
        else:
            print(str(case["reference_note"]).strip())

        # 2. Print Predicted Output (What your LAOS AI generated)
        print("\n\n🤖 PREDICTED OUTPUT (LAOS Generated JSON):")
        print("-" * 50)
        print(case["generated_note"].strip())

        print("\n" + " "*35 + "⬇️") # Visual separator between cases

except FileNotFoundError:
    print(f"❌ Could not find {RESULTS_PATH}. Make sure your evaluation cell finished running and saved the file!")

✅ Successfully loaded 57 cases for review.

 🏥 LAOS PIPELINE: GROUND TRUTH VS. PREDICTED OUTPUTS


🩺 CASE 1 / 57

🧑‍⚕️ GROUND TRUTH (Real Doctor's Note):
--------------------------------------------------
{"day": 1, "consultation": 1, "presenting_complaint": "I've been having really bad diarrhea for the last 3 days", "note": "3/7 hx of diarrhea, mainly watery. No blood in stool. Opening bowels x6/day. Associated LLQ pain - crampy, intermittent, nil radiation.  Also vomiting - mainly bilous. No blood in vomit. Fever on first day, nil since. Has been feeling lethargic and weak since. \n\nTakeaway 4/7 ago - Chinese restaurant. Wife and children also unwell with vomiting, but no diarrhea. No other unwell contacts. \n\nPMH: Asthma\nDH: Inhalers\nSH: works as an accountant. Lives with wife and children. Affecting his ADLs as has to be near toilet often. \nNil smoking/etOH hx\n\nImp: gastroenteritis \n\nPlan:\nConservative management - rest, push fluids, paracetamol if feverish. Recommend OTC

In [ ]:
import shutil
from IPython.display import FileLink, display

# 1. Zip the model folder
zip_name = "/kaggle/working/llama3_clinical_raft_final"
folder_to_zip = "/kaggle/working/models/lora_adapters/llama3_clinical_raft_v1"
shutil.make_archive(zip_name, 'zip', folder_to_zip)

# 2. Generate the clickable download link
print("✅ Model zipped successfully! Click the link below to download:")
display(FileLink("llama3_clinical_raft_final.zip"))

✅ Model zipped successfully! Click the link below to download:


/kaggle/working/llama3_clinical_raft_final.zip